# mcp

> Route stdio MCP requests to jupygate gateways

In [ ]:
#| default_exp mcp

Claude Code starts `clikernel-mcp` for each conversation. Its `Router` accepts MCP messages over stdio and sends tool calls to jupygate gateways. Jupygate implements the tools. The router gets their schemas from the local gateway and adds `host` to `list_kernels`, `use_kernel`, and `create`.

Each `(host, agent)` pair has its own gateway session, with `JUPYGATE_AGENT` passed in subagent `initialize` defaults. Calls without an explicit `host` go to their caller's current host; a subagent starts on the main conversation's current host. On a subagent's first `exec` with no current kernel, the router lazily creates a kernel matching the main session's current kernel implementation. Idle subagent sessions close with `DELETE /mcp` after 30 minutes. The router removes `host` and `agent` before forwarding a call while preserving JSON-RPC request ids and routing cancellations to the caller's session.

Before forwarding `exec`, the router intercepts `%cont` to page through truncated replies and checks all other cells against `clikernel.rules`. Most rules add a note to the reply. Two rules block the cell, which then never reaches a gateway. Replies over `maxout` UTF-8 bytes are saved to a temporary file and returned in chunks.

`main` closes its gateway sessions on EOF, SIGTERM, or an exception. Closing a session sends an HTTP DELETE. The gateway then stops the kernels that session created with autoclose. If the router started a child gateway, it stops that process too, and removes any temporary output directory.


In [ ]:
import asyncio, httpx, json, re, signal, tempfile, time
from contextlib import suppress
from fastcore.utils import *
from fastcore.script import call_parse, store_true
from mcpmini.core import serve_stdio, jresp, jerr
from clikernel.core import DEFAULT_URL, Gateway, default_gateway, resolve, session_defaults
from clikernel.rules import scan
from clikernel import __version__

In [ ]:
from fastcore.test import *
from itertools import count
from IPython.display import Markdown
import re, tempfile
from mcpmini.core import jreq, jtool
from jupyasyncclient import start_gateway

## Gateway sessions

In [ ]:
HOST_PARAM = {'type': 'string', 'description': 'Gateway to target: a gateways.toml name, or empty for the default local gateway'}
HOSTED = ('list_kernels', 'use_kernel', 'create')
MAXOUT = 40_000
AGENT_IDLE = 1800
AGENT_KERNEL = "New kernel for this subagent: names from any other kernel, the parent's included, aren't defined here."
_CONT_RE = re.compile(r'^%cont(?:\s+(\S+))?$')

class Router:
    "Route stdio MCP to jupygate with one backend session per `(host, agent)` and a current host for each caller."
    def __init__(
        self,
        cfgdir=None,  # Config dir for `session_defaults` and `gateways.toml` (the standard one if None)
        quiet=False,  # Keep startup output and rule notes out of replies?
        maxout=MAXOUT,  # Max UTF-8 bytes per text reply (0 or None disables capping)
        url=DEFAULT_URL,  # Local gateway URL to try before spawning a child (ignored when `CLIKERNEL_HOST` is set)
        agent_idle=AGENT_IDLE,  # Seconds before closing an idle subagent session (0 or None disables)
    ):
        self.cfgdir,self.quiet,self.maxout,self.url,self.agent_idle = cfgdir,quiet,maxout,url,agent_idle
        self.sessions,self.cur,self.inflight,self.child = {},{},{},None
        self._tmpdir,self._out_seq,self._conts,self._last = None,0,{},{}
        self._init_locks,self._agent_locks,self._agent_gen,self._last_used,self._reaper = {},{},{},{},None

    async def session(self, host='', agent=None):
        "Return the session for `(host, agent)`, initializing it on first use. Empty `host` selects the default local gateway."
        key = (host, agent)
        if key not in self.sessions:
            async with self._init_locks.setdefault(key, asyncio.Lock()):
                if key not in self.sessions:
                    if not host and agent is not None: await self.session('')
                    d = session_defaults(self.cfgdir, self.quiet, local=not host and not os.environ.get('CLIKERNEL_HOST'))
                    if agent is not None: d.setdefault('env', {})['JUPYGATE_AGENT'] = agent
                    if host: url, token, verify = resolve(host, self.cfgdir)
                    elif self.child: url, token, verify = self.child.url, None, True
                    elif ch := os.environ.get('CLIKERNEL_HOST'): url, token, verify = resolve(ch, self.cfgdir)
                    elif agent is None:
                        self.sessions[key], self.child = await default_gateway(self.cfgdir, self.quiet, url=self.url)
                        return self.sessions[key]
                    else: url, token, verify = self.url, os.environ.get('CLIKERNEL_TOKEN'), True
                    self.sessions[key] = await Gateway(url, token, verify).initialize(d)
                    if agent is not None:
                        self._last_used[key] = time.monotonic()
                        if self.agent_idle and (self._reaper is None or self._reaper.done()):
                            self._reaper = asyncio.create_task(self._reap_loop())
        return self.sessions[key]

    async def _reap_loop(self):
        while any(a is not None for _, a in self.sessions):
            await asyncio.sleep(max(0.05, min(60.0, self.agent_idle / 2)))
            now, active = time.monotonic(), set(self.inflight.values())
            stale = [k for k, s in list(self.sessions.items())
                if k[1] is not None and s not in active and now - self._last_used.get(k, now) >= self.agent_idle]
            for k in stale:
                s = self.sessions.pop(k, None)
                for m in (self._last_used, self._init_locks, self._agent_locks, self._agent_gen): m.pop(k, None)
                if s:
                    with suppress(Exception): await s.aclose()

The examples start a disposable gateway on a free local port. A temporary `startup.py` defines `base = 42` and prints `ready`. `session()` opens the default local gateway session on first use. The gateway supplies its kernel-selection instructions during initialization.


In [ ]:
g = start_gateway()
os.environ.pop('CLIKERNEL_HOST', None)
tmp = tempfile.TemporaryDirectory()
cfgd = Path(tmp.name)
(cfgd/'startup.py').write_text('base = 42; print("ready")')
router = Router(cfgd, url=g.url)
local = await router.session()
test_eq(local.url, g.url)
local.info['instructions']

"Select a kernel with create(kernel=...) or use_kernel, then run code with exec(code=...). Kernel implementations are listed in create's schema; creation and selection return usage guidance. New kernels require an explicit implementation; exec never creates or switches kernels. An optional dlgname binds creation to a dialog or overrides execution for one call. Kernels this session creates autoclose by default. Python startup/inspectors run only in Python kernels."

`tools` takes the gateway's schemas, adds `host` to `list_kernels`, `use_kernel`, and `create`, and removes the local `enum` from `create`'s `kernel` parameter so clients can request remote-only kernelspecs. Execution follows the caller's current host instead of accepting a host of its own.

In [ ]:
@patch
async def tools(self:Router):
    "Return the local gateway's tools with `host` added to `list_kernels`, `use_kernel`, and `create`, and `create.kernel` unconstrained for remote hosts."
    ts = await (await self.session()).tools()
    for t in ts:
        props = t['inputSchema'].setdefault('properties', {})
        if t['name'] in HOSTED: props['host'] = dict(HOST_PARAM)
        if t['name'] == 'create' and 'kernel' in props: props['kernel'].pop('enum', None)
    return ts

These are the tools whose schemas let the client choose a host:

In [ ]:
listed = await router.tools()
byname = {t['name']: t for t in listed}
assert all('host' in byname[n]['inputSchema']['properties'] for n in HOSTED)
assert 'host' not in byname['exec']['inputSchema']['properties']
assert 'enum' not in byname['create']['inputSchema']['properties']['kernel']
[n for n,t in byname.items() if 'host' in t['inputSchema']['properties']]


['list_kernels', 'create', 'use_kernel']

## Dispatching requests

Claude Code disconnects a stdio server that sends a message over 16 MB. Jupygate limits the text of a reply to 1 MB. It sends images whole, because other gateway clients need large images. The router limits the images in each tool result to 5 MB of base64 data. `_cap_images` keeps each image that fits in what remains of that budget. It replaces every other image with a line of text giving its type and size.

In [ ]:
def _cap_images(content, budget=5 << 20):
    "`content`, with each image that would take the total base64 image data past `budget` bytes replaced by a line of text."
    left,res = budget,[]
    for b in content:
        if b['type'] == 'image':
            if (n := len(b['data'])) > left: b = dict(type='text', text=f"[{b['mimeType']} image of {n:,} bytes omitted: images in one reply are limited to {budget:,} bytes]")
            else: left -= n
        res.append(b)
    return res

With a budget of 4 bytes, the 3-byte image fits and the 2-byte image doesn't. The 1-byte image still fits:

In [ ]:
imgs = [dict(type='image', mimeType='image/png', data='x'*n) for n in (3, 2, 1)]
capped = _cap_images(imgs, budget=4)
test_eq([b['type'] for b in capped], ['image', 'text', 'image'])
capped[1]['text']

Some MCP hosts spill large tool replies to disk with no inline preview or truncate them in the middle; Antigravity does both above 10,240 UTF-8 bytes (including its timestamp header). `_cut_utf8` and `_page` slice UTF-8 bytes to fit a `budget` along character boundaries (snapping back to a nearby newline), reserving space for a truncation footer and any `𝍁…𝍁` preservation wrapper (`full=True`, such as `doc()`), with the closing `𝍁` placed after the footer so transcript compactors keep every chunk intact.

In [ ]:
def _cut_utf8(b, budget, snap=256):
    "Byte index <= `budget` in UTF-8 bytes `b` on a char boundary, snapping back to a nearby newline."
    end = min(budget, len(b))
    while end > 0 and end < len(b) and (b[end] & 0xC0) == 0x80: end -= 1
    if end == 0 and b:
        end = 1
        while end < len(b) and (b[end] & 0xC0) == 0x80: end += 1
    elif end < len(b):
        nl = b.rfind(b'\n', max(end // 2, end - snap), end)
        if nl >= 0: end = nl + 1
    return end

def _page(data, offset, budget, mk_footer, full=False):
    "Return `(text, new_offset)` fitting `data[offset:]` plus footer and `𝍁…𝍁` wrapper within `budget` UTF-8 bytes."
    wrap = 8 if full else 0
    tail = data[offset:]
    if len(tail) + wrap <= budget:
        body = tail.decode('utf-8')
        return (f'𝍁{body}𝍁' if full else body), len(data)
    avail = budget - wrap - len(mk_footer(len(tail)).encode('utf-8'))
    end = _cut_utf8(tail, max(1, avail))
    body = tail[:end].decode('utf-8') + mk_footer(len(tail) - end)
    return (f'𝍁{body}𝍁' if full else body), offset + end

`_page` enforces the UTF-8 byte budget on multi-byte text, snaps to line breaks, and puts the closing `𝍁` after the footer:

In [ ]:
sample = ('café line\n' * 12).encode('utf-8')
chunks, off = [], 0
while off < len(sample):
    ch, off = _page(sample, off, 70, lambda rem: f'\n[{rem} left]', full=True)
    assert len(ch.encode('utf-8')) <= 70 and ch.startswith('𝍁') and ch.endswith('𝍁')
    chunks.append(ch)
assert len(chunks) > 2 and chunks[0].endswith(' left]𝍁') and not chunks[-1].endswith(' left]𝍁')
test_eq(''.join(re.sub(r'\n\[\d+ left\]$', '', c[1:-1]) for c in chunks), sample.decode('utf-8'))
len(chunks)

When a reply exceeds `maxout` UTF-8 bytes (default 40,000; `0` disables capping), `_cap_text` writes the complete unwrapped text to a lazy per-router temporary directory and returns the first chunk with a footer naming that file, the remaining byte count, and `%cont` (or `%cont <id>`). `_cont` pages from the saved file, keyed per `agent` so subagents and parallel calls stay isolated.

In [ ]:
@patch
def _cap_text(self:Router, content, agent, note=''):
    "Cap text blocks in `content` (plus optional `note`) to `self.maxout` UTF-8 bytes, saving overflow for `%cont`."
    note_blk = [dict(type='text', text=note)] if note else []
    if not self.maxout: return note_blk + content
    joined = ''.join(b.get('text', '') for b in content if b['type'] == 'text')
    note_bytes = len(note.encode('utf-8')) + 1 if note else 0
    if len(joined.encode('utf-8')) + note_bytes <= self.maxout: return note_blk + content
    full = joined.startswith('𝍁') and joined.endswith('𝍁') and len(joined) >= 2
    data = (joined[1:-1] if full else joined).encode('utf-8')
    if self._tmpdir is None: self._tmpdir = tempfile.TemporaryDirectory(prefix='clikernel-')
    self._out_seq += 1
    oid = str(self._out_seq)
    path = Path(self._tmpdir.name)/f'out_{oid}.txt'
    path.write_bytes(data)
    total = len(data)
    mk_foot = lambda rem: f'\n[truncated: {rem:,} of {total:,} bytes left; full output: {path}; %cont or %cont {oid} for next]'
    chunk, offset = _page(data, 0, self.maxout - note_bytes, mk_foot, full=full)
    if offset < total:
        self._conts.setdefault(agent, {})[oid] = (path, offset, full, total)
        self._last[agent] = oid
    return note_blk + [dict(type='text', text=chunk)] + [b for b in content if b['type'] != 'text']

In [ ]:
@patch
def _cont(self:Router, agent, oid=None):
    "Return the next chunk of a truncated reply for `agent`, optionally selecting `oid`."
    aconts = self._conts.get(agent) or {}
    key = oid or self._last.get(agent)
    if not key or key not in aconts:
        msg = f'No pending output with id {oid}.' if oid else 'No pending output to continue.'
        return dict(isError=True, content=[dict(type='text', text=msg)])
    path, offset, full, total = aconts[key]
    data = path.read_bytes()
    mk_foot = lambda rem: f'\n[{rem:,} of {total:,} bytes left; %cont or %cont {key} for next]'
    chunk, new_offset = _page(data, offset, self.maxout, mk_foot, full=full)
    if new_offset >= total:
        aconts.pop(key, None)
        if self._last.get(agent) == key: self._last.pop(agent, None)
    else:
        aconts[key] = (path, new_offset, full, total)
        self._last[agent] = key
    return dict(content=[dict(type='text', text=chunk)])

In [ ]:
def _full_body(t): return t[1:-1] if len(t) >= 2 and t[0] == '𝍁' and t[-1] == '𝍁' else None

def _stack(blocks):
    "Combine MCP reply blocks in order, keeping `𝍁…𝍁` around the combined text if any text block is marked full."
    out = [dict(b) for b in blocks]
    for b in out[:-1]:
        if (t := b.get('text')) and not t.endswith('\n'): b['text'] = t + '\n'
    if any((t := b.get('text')) is not None and _full_body(t.rstrip('\n')) is not None for b in out):
        parts, rest = [], []
        for b in out:
            if (t := b.get('text')) is None: rest.append(b)
            else:
                body = t.rstrip('\n')
                inner = _full_body(body)
                parts.append((body if inner is None else inner) + t[len(body):])
        return [dict(type='text', text='𝍁' + ''.join(parts) + '𝍁'), *rest]
    return out

def _is_no_current(res):
    r = res.get('result') if res else None
    return bool(r and r.get('isError') and 'no current kernel' in ''.join(b.get('text', '') for b in r.get('content', [])))

@patch
async def _send(self:Router, h, agent, msg):
    key, rid = (h, agent), msg['id']
    s = await self.session(h, agent)
    self.inflight[rid] = s
    if agent is not None: self._last_used[key] = time.monotonic()
    try:
        try: return s, await s.tr.send(msg)
        except httpx.HTTPStatusError as e:
            if e.response.status_code != 404: raise
            self.sessions.pop(key, None)
            s = await self.session(h, agent)
            self.inflight[rid] = s
            return s, await s.tr.send(msg)
    finally:
        if agent is not None and key in self.sessions: self._last_used[key] = time.monotonic()

@patch
async def _parent_kernel(self:Router, h):
    s = self.sessions.get((h, None))
    if not s: return None
    try: r = await s.call('list_kernels')
    except httpx.HTTPStatusError as e:
        if e.response.status_code != 404: raise
        self.sessions.pop((h, None), None)
        return None
    if r.get('isError'): return None
    txt = ''.join(b.get('text', '') for b in r.get('content', []) if b.get('type') == 'text')
    for line in txt.splitlines():
        if line.endswith('<- current') and (m := re.search(r'\bkernel=(\S+)', line)): return m.group(1)
    return None

@patch
async def _ensure_agent_kernel(self:Router, h, agent, s, cwd, gen):
    key = (h, agent)
    async with self._agent_locks.setdefault(key, asyncio.Lock()):
        if self._agent_gen.get(key, 0) != gen: return []
        impl = await self._parent_kernel(h)
        if not impl: return None
        kw = dict(kernel=impl)
        if cwd: kw['cwd'] = cwd
        r = await s.call('create', **kw)
        if r.get('isError'): return r
        self._agent_gen[key] = gen + 1
        return [dict(type='text', text=AGENT_KERNEL), *r['content']]

`dispatch` answers `initialize` and `ping` locally. It strips `host` and `agent` and routes each `(host, agent)` to its own gateway session, forwarding tool calls with their original JSON-RPC ids. `create` and `use_kernel` change the caller's current host when given an explicit `host` and the call succeeds; failed selections and `list_kernels` leave it unchanged. Any hook-supplied `cwd` is forwarded to the automatic local gateway and dropped before forwarding to named hosts or an explicit `CLIKERNEL_HOST`. When a subagent calls `exec` without a current kernel, `dispatch` serializes lazy `create` using the main session's current kernel implementation, prepends a notice, and retries `exec`.

For `exec`, `dispatch` first handles `%cont` (and `%cont <id>`) locally to return the next chunk of a truncated reply. All other `exec` cells run through the cell rules. A blocking rule returns an error result without contacting a gateway. Notes go in a text block before the gateway's content, unless the router is quiet.

`dispatch` passes every tool result through `_cap_images` and `_cap_text`, described above.

A gateway that restarted answers 404 for a session id it no longer knows. `dispatch` then opens a new session on that host and sends the tool call once more. The new session has no current kernel.

Cancellation notifications go to the gateway session handling the request, with their original request ids. A cancellation for a request that has finished goes nowhere.

In [ ]:
@patch
async def dispatch(self:Router, msg, requester=None):
    "Answer initialization and ping locally. Forward tool calls to gateways."
    method,id = msg.get('method'), msg.get('id')
    try:
        if method == 'initialize':
            info = (await self.session()).info
            return jresp(id, dict(protocolVersion=msg['params'].get('protocolVersion', '2025-06-18'), capabilities=dict(tools={}),
                serverInfo=dict(name='clikernel', version=__version__), instructions=info.get('instructions')))
        if id is None:
            if method == 'notifications/cancelled' and (s := self.inflight.get(msg['params'].get('requestId'))) is not None:
                await s.tr.send(msg)
            return None
        if method == 'ping': return jresp(id, {})
        if method == 'tools/list': return jresp(id, dict(tools=await self.tools()))
        if method == 'tools/call':
            args = msg['params'].setdefault('arguments', {})
            agent = args.pop('agent', None) or None
            tool = msg['params']['name']
            if tool == 'exec' and (m := _CONT_RE.match(args.get('code', '').strip())): return jresp(id, self._cont(agent, m.group(1)))
            fired = scan(args.get('code', '')) if tool == 'exec' else []
            if blocks := [r.note for r in fired if r.block]:
                return jresp(id, dict(isError=True, content=[dict(type='text', text=' '.join(blocks)+' (If the block seems wrong here, tell your user.)')]))
            has_host = 'host' in args
            host = args.pop('host', '') or ''
            h = host if has_host else self.cur.setdefault(agent, self.cur.get(None, ''))
            if h or os.environ.get('CLIKERNEL_HOST'): args.pop('cwd', None)
            cwd = args.pop('cwd', None) if tool == 'exec' else None
            gen = self._agent_gen.get((h, agent), 0) if agent is not None else 0
            try:
                s, res = await self._send(h, agent, msg)
                if agent is not None and tool == 'exec' and _is_no_current(res):
                    opened = await self._ensure_agent_kernel(h, agent, s, cwd, gen)
                    if isinstance(opened, dict): res = jresp(id, opened)
                    elif opened is not None:
                        _, res = await self._send(h, agent, msg)
                        if opened and res and 'result' in res and not res['result'].get('isError'):
                            res['result']['content'] = _stack(opened + res['result']['content'])
            finally: self.inflight.pop(id, None)
            if res and 'result' in res:
                if has_host and tool in ('use_kernel', 'create') and not res['result'].get('isError'):
                    self.cur[agent] = host
                content = _cap_images(res['result']['content'])
                note = ''.join(f'<note>\n{r.note}\n</note>\n' for r in fired) if fired and not self.quiet else ''
                res['result']['content'] = self._cap_text(content, agent, note=note)
            return res
        return jerr(id, -32601, f'method not found: {method}')
    except Exception as e: return None if id is None else jerr(id, -32603, str(e))

The stdio client initializes its own session with the router. The reply uses the client's protocol version and includes the local gateway's instructions:

In [ ]:
init = await router.dispatch(jreq('initialize', 1, protocolVersion='2025-11-25', capabilities={}, clientInfo=dict(name='demo', version='0')))
test_eq(init['result']['serverInfo']['name'], 'clikernel')
test_eq(init['result']['protocolVersion'], '2025-11-25')
Markdown(init['result']['instructions'])

Select a kernel with create(kernel=...) or use_kernel, then run code with exec(code=...). Kernel implementations are listed in create's schema; creation and selection return usage guidance. New kernels require an explicit implementation; exec never creates or switches kernels. An optional dlgname binds creation to a dialog or overrides execution for one call. Kernels this session creates autoclose by default. Python startup/inspectors run only in Python kernels.

The remaining lessons use `call` for an MCP tool result and `text` for its text content. These notebook helpers assign request ids and remove the JSON-RPC envelope.

In [ ]:
#| hide
request_ids = count(2)
def txt(r): return ''.join(c.get('text', '') for c in r['content'] if c['type'] == 'text')
async def call(name, **args): return (await router.dispatch(jtool(name, next(request_ids), **args)))['result']
async def text(name, **args):
    r = await call(name, **args)
    assert not r.get('isError'), txt(r)
    return txt(r)

Creating a Python kernel runs `startup.py`. The next execution sees the value it defined:

In [ ]:
created = await text('create', kernel='py')
assert 'created kernel' in created and 'ready' in created
base = await text('exec', code='base')
test_eq(base, '42')
base

'42'

`gateways.toml` maps host names to gateways. Here, `alt` refers to a second disposable gateway. Creating a kernel there makes it the caller's current host. The following Python calls omit `host` and run in that kernel:

In [ ]:
g2 = start_gateway()
(cfgd/'gateways.toml').write_text(f'[gateways.alt]\nurl = "{g2.url}"\n')
created = await text('create', kernel='py', host='alt')
assert 'created kernel' in created and 'ready' in created
await text('exec', code='marker = 7')
marker = await text('exec', code='marker')
test_eq(marker, '7')
cwd_dir = (Path(tmp.name)/'proj').resolve()
cwd_dir.mkdir()
await text('create', kernel='py', host='', agent='cwd-test', cwd=str(cwd_dir))
test_eq(await text('exec', code='import os; os.getcwd()', agent='cwd-test'), str(cwd_dir))
# Remote hosts drop local `cwd` so a local path never breaks a remote kernel.
await text('create', kernel='py', host='alt', agent='cwd-test', cwd='/nonexistent/local/path')
test_ne(await text('exec', code='import os; os.getcwd()', agent='cwd-test'), '/nonexistent/local/path')
with modified_env(CLIKERNEL_HOST=g.url):
    await text('create', kernel='py', host='', agent='cwd-test', cwd='/nonexistent/local/path')
test_ne(await text('exec', code='import os; os.getcwd()', agent='cwd-test'), '/nonexistent/local/path')
marker

'7'

Use `host=''` to address the default local gateway. Listing its kernels or failing a selection doesn't change the current host. Selecting its kernel with `use_kernel` does.

The local kernel has `base` from startup but no `marker`. The `alt` gateway still has its kernel:

In [ ]:
kid = re.search(r'^(\w+) .*<- current$', await text('list_kernels', host=''), re.M).group(1)
test_eq(await text('exec', code='marker'), '7')
assert (await call('use_kernel', kernel='nonexistent', host='')).get('isError')
test_eq(await text('exec', code='marker'), '7')
await text('use_kernel', kernel=kid, host='')
assert 'NameError' in await text('exec', code='marker')
test_eq(await text('exec', code='base'), '42')
await text('list_kernels', host='alt')

'c722209268df41aca150ce6477304a7c  alive  kernel=ipymini  language=python  connections=1  dlgname=far.ipynb  <- current'

Jupygate returns this PIL image as a text block followed by an MCP image block. It follows `fastcore.nbio.IMG_MIMES` when choosing a format. PIL supplies PNG and JPEG representations, and that preference order selects JPEG. The router passes both content blocks to the client unchanged:


In [ ]:
r = await call('exec', code=r'''
from PIL import Image
Image.new("RGB", (300, 200), "red")
''')
blocks = r['content']
test_eq([b['type'] for b in blocks], ['text', 'image'])
test_eq(blocks[1]['mimeType'], 'image/jpeg')
{k:v for k,v in blocks[1].items() if k != 'data'}


{'type': 'image', 'mimeType': 'image/jpeg'}

## Cell rules

Before forwarding an `exec` call, the router checks its code against the rules in `clikernel.rules`. Each rule catches a mistake agents often make with this toolkit, such as computing exhash addresses or reading files with `read_text()`. Most rules add a note to the reply, and the cell still runs. This cell puts a backslash escape in a string that isn't raw:

In [ ]:
strcell = r'''s = """a\nb"""
len(s)'''
noted = await text('exec', code=strcell)
assert noted.startswith('<note>') and noted.endswith('3')
PrettyString(noted)

Two rules block the cell instead: shell access through `subprocess`, `os.system` or `os.popen`, and changes to `sys.path`. A blocked cell never reaches the gateway, so the kernel is unchanged. The router can't see which kernel will run the code, so rules match only Python forms that other kernel languages don't produce. That is why a `!` shell escape isn't blocked: APL's `!5` (factorial) parses the same way.

In [ ]:
blocked = await call('exec', code='import subprocess')
assert blocked['isError']
test_eq(await text('exec', code="'subprocess' in dir()"), 'False')
txt(blocked)

`--quiet` sets the router's `quiet` attribute. It leaves the notes out of replies, but blocking rules still apply:

In [ ]:
router.quiet = True
test_eq(await text('exec', code=strcell), '3')
assert (await call('exec', code='import subprocess'))['isError']
router.quiet = False

## Output capping and `%cont`

Lower `router.maxout` to see text capping in action. A reply over `maxout` UTF-8 bytes is written in full to a temporary file and returned as a bounded first chunk:


In [ ]:
router.maxout = 350
expected = ''.join(f'row {i:02d} café\n' for i in range(40))
first_chunk = await text('exec', code="print(''.join(f'row {i:02d} café\\n' for i in range(40)), end='')")
assert len(first_chunk.encode('utf-8')) <= 350
m = re.search(r'full output: (\S+); %cont or %cont (\d+)', first_chunk)
saved, oid1 = Path(m.group(1)), m.group(2)
test_eq(saved.read_text(), expected)
PrettyString(first_chunk)

`full=True` replies such as `doc()` keep `𝍁` at both ends of every chunk (with the closing `𝍁` after the footer), while the saved file holds the unwrapped text:

In [ ]:
doc_chunk = await text('exec', code="from pyskills.core import doc; doc(doc)")
assert len(doc_chunk.encode('utf-8')) <= 350 and doc_chunk.startswith('𝍁') and doc_chunk.endswith(']𝍁')
doc_saved = Path(re.search(r'full output: (\S+);', doc_chunk).group(1))
assert not doc_saved.read_text().startswith('𝍁')
PrettyString(doc_chunk)

`%cont <id>` selects an earlier overflow after a second one arrives, `%cont` continues it to completion, another `agent` sees isolated state, and `%%cont` passes through untouched:

In [ ]:
other = await call('exec', code='%cont', agent='sub-other')
assert other.get('isError') and 'No pending output' in txt(other)
parts, p = [re.sub(r'\n\[truncated: .*\]$', '', first_chunk)], await text('exec', code=f'%cont {oid1}')
while re.search(r'\n\[[\d,]+ of [\d,]+ bytes left;', p):
    assert len(p.encode('utf-8')) <= 350
    parts.append(re.sub(r'\n\[[\d,]+ of [\d,]+ bytes left;.*\]$', '', p))
    p = await text('exec', code='%cont')
test_eq(''.join(parts + [p]), expected)
assert 'UsageError' in txt(await call('exec', code='%%cont\npass'))
tmpdir, router.maxout = Path(router._tmpdir.name), MAXOUT

A subagent identified by `agent` gets its own gateway session (with `JUPYGATE_AGENT` in `initialize` defaults) and lazily creates an isolated kernel matching the main session's implementation on its first `exec`, serializing concurrent first calls. Deleting the subagent's kernel lets its next `exec` start a fresh one, and an idle subagent session closes via `DELETE /mcp` after `agent_idle` seconds without disturbing the main session:

In [ ]:
await text('exec', code='x = 1')
r1, r2 = await asyncio.gather(text('exec', code='x = 2; base + x', agent='sub1'),
    text('exec', code='import os; print(os.environ["JUPYGATE_AGENT"], end="")', agent='sub1'))
assert sum('New kernel for this subagent' in r and 'created kernel ' in r for r in (r1, r2)) == 1
assert r1.endswith('44') and r2.endswith('sub1')
test_eq(await text('exec', code='x'), '1')
test_eq(await text('exec', code='import os; print(os.environ.get("JUPYGATE_AGENT", "unset"), end="")'), 'unset')
assert (await text('restart', agent='sub1')).startswith('restarted kernel ')
test_eq(await text('exec', code='import os; print(os.environ["JUPYGATE_AGENT"], end="")', agent='sub1'), 'sub1')
await text('delete_kernel', agent='sub1')
recreated = await text('exec', code='from pyskills.core import doc; doc(doc)', agent='sub1')
assert recreated.startswith('𝍁New kernel for this subagent') and 'created kernel ' in recreated and recreated.endswith('𝍁')
test_eq(await text('exec', code='x'), '1')

## Closing sessions

`aclose` attempts every session close, stops an owned child gateway, and removes any temporary output directory even if a close fails. It raises the session errors together after cleanup.

In [ ]:
@patch
async def aclose(self:Router):
    "Close all sessions and their autoclose kernels, remove any temporary output directory, then stop any owned gateway."
    if self._reaper and not self._reaper.done():
        self._reaper.cancel()
        with suppress(asyncio.CancelledError): await self._reaper
    sessions = list(self.sessions.values())
    self.sessions.clear()
    try: results = await asyncio.gather(*(s.aclose() for s in sessions), return_exceptions=True)
    finally:
        if self.child:
            self.child.stop()
            self.child = None
        if self._tmpdir:
            self._tmpdir.cleanup()
            self._tmpdir = None
    errors = [r for r in results if isinstance(r, BaseException)]
    if errors: raise BaseExceptionGroup('MCP session cleanup failed', errors)

An idle subagent session closes via `DELETE /mcp` after `agent_idle` seconds, stopping its autoclose kernel while leaving the main session untouched:

In [ ]:
idle_r = Router(cfgd, url=g.url, agent_idle=0.15)
try:
    await idle_r.dispatch(jtool('create', 1, kernel='py'))
    out = txt((await idle_r.dispatch(jtool('exec', 2, code='1 + 1', agent='ephem')))['result'])
    ekid = re.search(r'created kernel (\w+)', out).group(1)
    assert ('', 'ephem') in idle_r.sessions
    for _ in range(30):
        if ('', 'ephem') not in idle_r.sessions: break
        await asyncio.sleep(0.05)
    assert ('', 'ephem') not in idle_r.sessions and ekid not in [k['id'] for k in httpx.get(f'{g.url}/api/kernels').json()]
    test_eq(txt((await idle_r.dispatch(jtool('exec', 3, code='base')))['result']), '42')
finally: await idle_r.aclose()

Ending the router's sessions stops the kernels it created with autoclose. Selecting an existing kernel does not make the router responsible for its lifetime.

Our gateways were already running when the router connected. It owns neither process. A new router can still connect to both gateways, but neither has any remaining kernels:

In [ ]:
await router.aclose()
assert router.child is None and not tmpdir.exists()
chk = Router(cfgd, url=g.url)
near = await (await chk.session()).text('list_kernels')
far = await (await chk.session('alt')).text('list_kernels')
await chk.aclose()
test_eq((near, far), ('no kernels', 'no kernels'))
near, far


('no kernels', 'no kernels')

## The stdio server

`main` serves one router over stdio. The router opens each gateway session on first use. Initialization opens the local session, starting a child gateway if necessary. A later `tools/list` request fetches the tool schemas.

On shutdown, `main` closes all gateway sessions, removes any temporary output directory, and stops any child gateway it owns. SIGTERM cancels the serving task through this same cleanup. SIGKILL cannot run cleanup; a resident gateway's `--cull-idle` setting controls how long abandoned kernels remain. `--quiet` suppresses kernel startup output and rule notes in replies. `--cfgdir` names the directory holding `startup.py` and `gateways.toml`, in place of `~/.config/clikernel/`. `--maxout` sets the per-reply UTF-8 byte cap (default 40,000; `0` disables capping).


In [ ]:
@call_parse
def main(
    quiet:store_true=False,  # Keep startup output and rule notes out of replies
    cfgdir:str=None,  # Directory holding `startup.py` and `gateways.toml`; `~/.config/clikernel/` if unset
    maxout:int=MAXOUT,  # Max UTF-8 bytes per text reply (0 disables capping)
):
    "Run `clikernel-mcp` as a stdio server."
    async def _main():
        router = Router(cfgdir, quiet=quiet, maxout=maxout)
        loop,task = asyncio.get_running_loop(),asyncio.current_task()
        signal.signal(signal.SIGTERM, lambda *_: loop.call_soon_threadsafe(task.cancel))
        try: await serve_stdio(router)
        finally: await router.aclose()
    asyncio.run(_main())


The installed `clikernel-mcp` command also accepts an MCP stdio client. Call `create` before `exec`. `--cfgdir` points it at the temporary config directory; the Python kernel starts with `base` from that `startup.py`. Attaching to an existing kernel with `use_kernel` leaves its lifetime to whoever created it: SIGTERM stops the router and the kernel it created while the attached kernel remains usable afterwards:

In [ ]:
import shutil
from mcpmini.core import MCPClient


In [ ]:
#| hide
cmd = shutil.which('clikernel-mcp')
assert cmd, 'clikernel-mcp script not installed'
env = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(cfgd)}

In [ ]:
ext = httpx.post(f'{g.url}/api/kernels', json=dict(name='py')).json()['id']
async with MCPClient.stdio([cmd, '--cfgdir', str(cfgd)], env=env) as m:
    assert 'exec' in dir(m.tools) and 'py' not in dir(m.tools) and 'lua' not in dir(m.tools)
    await m.tools.create(kernel='py')
    res = await m.tools.exec(code='base')
    await m.tools.use_kernel(kernel=ext)
    await m.tools.exec(code='attached = 99')
    m.tr.p.terminate()
    await m.tr.p.wait()
test_eq(res, '42')
test_eq([k['id'] for k in httpx.get(f'{g.url}/api/kernels').json()], [ext])
chk = await Gateway(g.url).initialize(session_defaults(cfgd))
try:
    await chk.text('use_kernel', kernel=ext)
    test_eq(await chk.text('exec', code='attached'), '99')
finally:
    await chk.aclose()
    httpx.delete(f'{g.url}/api/kernels/{ext}')
res


{'Python': '42', 'Luau': '42'}

## Subagent kernels

Claude Code sends the tool calls of every subagent through the conversation's one MCP connection, so by default its subagents all share the current kernel. A `PreToolUse` hook sees which agent made each call: a subagent's call carries `agent_id`, and the main conversation's calls don't. `clikernel-hook` adds that id to a subagent's clikernel calls as the argument `agent`. The router strips `agent`, opens a separate backend MCP session per `(host, agent)` with `JUPYGATE_AGENT` in its `initialize` defaults, lazily creates the subagent's kernel on first `exec` matching the main session's kernel implementation, and closes idle subagent sessions with `DELETE /mcp` after 30 minutes.

In [ ]:
def agent_input(
    event:dict,  # A Claude Code `PreToolUse` hook event
):
    "`event`'s tool input with `agent` set to the calling subagent's id, or None for a call from the main conversation."
    if aid := event.get('agent_id'): return event['tool_input'] | dict(agent=aid)

A subagent's `exec` gains `agent`. The main conversation's call stays as it is:

In [ ]:
ev = dict(hook_event_name='PreToolUse', tool_name='mcp__clikernel__exec', tool_input=dict(code='6*7'))
test_is(agent_input(ev), None)
sub = agent_input(ev | dict(agent_id='a3ccb4b3559e06c4c', agent_type='fork'))
test_eq(sub, dict(code='6*7', agent='a3ccb4b3559e06c4c'))
sub

`hook` runs `clikernel-hook`. It reads a `PreToolUse` event on stdin and prints the hook response: for Claude Code, `updatedInput` on a subagent's call (nothing for the main conversation); for Antigravity, whose language server shares one `clikernel-mcp` process across conversations and subagents and spawns plugin servers in the plugin directory, `overwrite` adding `agent=conversationId` and (when present) `cwd` on `clikernel` calls, and `decision='ask'` on other MCP servers.

In [ ]:
def _hook_response(event:dict):
    if 'toolCall' in event:
        tc = event['toolCall']
        args, cid = tc.get('args') or {}, event.get('conversationId')
        cwd = event.get('cwd') or first(event.get('workspacePaths') or ()) or ''
        extra = {k: v for k, v in dict(agent=cid, cwd=cwd).items() if v}
        if tc.get('name') == 'call_mcp_tool':
            srv = args.get('ServerName') or ''
            if extra and (srv == 'clikernel' or srv.endswith('_clikernel')):
                return dict(decision='allow', overwrite=dict(Arguments=(args.get('Arguments') or {}) | extra))
        elif extra: return dict(decision='allow', overwrite=extra)
        return dict(decision='ask')
    if upd := agent_input(event):
        return dict(hookSpecificOutput=dict(hookEventName='PreToolUse', updatedInput=upd))

def hook():
    "Run `clikernel-hook`: read a `PreToolUse` event on stdin and print the hook response giving each caller its own kernel."
    if resp := _hook_response(json.load(sys.stdin)): print(json.dumps(resp))

In [ ]:
test_is(_hook_response(ev), None)
test_eq(_hook_response(ev | dict(agent_id='sub1'))['hookSpecificOutput']['updatedInput'], dict(code='6*7', agent='sub1'))
agy = dict(conversationId='conv1', toolCall=dict(name='call_mcp_tool',
    args=dict(ServerName='aai-coding_clikernel', ToolName='exec', Arguments=dict(code='6*7'))))
test_eq(_hook_response(agy), dict(decision='allow', overwrite=dict(Arguments=dict(code='6*7', agent='conv1'))))
test_eq(_hook_response(agy | dict(cwd='/proj', workspacePaths=['/other'])),
    dict(decision='allow', overwrite=dict(Arguments=dict(code='6*7', agent='conv1', cwd='/proj'))))
test_eq(_hook_response(agy | dict(cwd='', workspacePaths=['/ws'])),
    dict(decision='allow', overwrite=dict(Arguments=dict(code='6*7', agent='conv1', cwd='/ws'))))
other = dict(conversationId='conv1', cwd='/proj', toolCall=dict(name='call_mcp_tool',
    args=dict(ServerName='other_mcp', ToolName='ping', Arguments={})))
test_eq(_hook_response(other), dict(decision='ask'))

Register the hook for the server's tools in Claude Code's settings, such as `~/.claude/settings.json`. The matcher uses the name the server was registered under:

```json
{"hooks": {"PreToolUse": [{"matcher": "mcp__clikernel__.*",
  "hooks": [{"type": "command", "command": "clikernel-hook"}]}]}}
```

Claude Code reads its settings at startup, so the hook applies from the next session.

In Antigravity, register `clikernel-hook` in `hooks.json` matching `call_mcp_tool` (for lazy-loaded servers, including plugin-namespaced names such as `aai-coding_clikernel`):

```json
{"clikernel-isolate": {"PreToolUse": [{"matcher": "call_mcp_tool",
  "hooks": [{"type": "command", "command": "/path/to/.venv/bin/clikernel-hook"}]}]}}
```

## A live client

These optional lessons use Claude Code's headless Agent SDK. They spend model tokens and have `#| eval: false` to exclude them from automated runs. Run them manually after changing the tools or updating Claude Code.

Shared setup restricts Claude to `create` and `exec`. The `ask_claude` helper returns its final answer and the tool names it used.


In [ ]:
#| eval: false
import logging, random
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage

In [ ]:
#| hide
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_tmp = tempfile.TemporaryDirectory()
live_dir = Path(live_tmp.name)
lenv = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': live_tmp.name}
opts = ClaudeAgentOptions(mcp_servers=dict(ck=dict(type='stdio', command=cmd, env=lenv)), cwd=live_tmp.name,
    allowed_tools=['mcp__ck__create', 'mcp__ck__exec'], max_turns=6)

async def ask_claude(prompt):
    msgs = [m async for m in query(prompt=prompt, options=opts)]
    used = {b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if isinstance(b, ToolUseBlock)}
    return first(m.result for m in msgs if isinstance(m, ResultMessage)), used

Claude must create a `py` kernel and use it to compute the result:

In [ ]:
#| eval: false
prompt = r'''Using the ck MCP tools, create a py kernel, then exec 17*19. Reply with just the number.'''
res, used = await ask_claude(prompt)
assert {'mcp__ck__create', 'mcp__ck__exec'} <= used
assert '323' in res
res


'323'

The next test checks image delivery through the router, stdio transport, and Claude Code's client. It chooses a random color without naming it in the prompt. The requested Python code reads the color from a file and displays a plain image. Claude must identify the color from that image. The final assertion compares its answer with the chosen color.

In [ ]:
#| eval: false
color = random.choice(['red', 'green', 'blue', 'yellow', 'purple', 'orange'])
(live_dir/'color.txt').write_text(color)
prompt = r'''
Using the ck MCP tools, create a py kernel, then run exactly this code with exec:

from PIL import Image
Image.new('RGB', (200,200), open('color.txt').read().strip())

Do not execute anything else or read color.txt any other way.
The exec result includes an image. Reply with just the color of that image.
'''
res2, _ = await ask_claude(prompt)
assert color in res2.lower()
color, res2

('purple', 'Purple.')

In [ ]:
#| hide
#| eval: false
live_tmp.cleanup()

In [ ]:
#| hide
g.stop()
g2.stop()
tmp.cleanup()


In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()